## Import Required Libraries

Import PySpark SQL functions, data types, Delta Lake utilities, and datetime for data processing and transformations.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, TimestampType, StringType
from delta.tables import DeltaTable
import datetime

## Configuration Variables

Define the catalog, schema, and table names for the bronze source, silver target, and data quality logging table.

In [0]:
CATALOG = "chicago_city"
SCHEMA = "silver"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_trips"   # fully qualified
DQ_TABLE = f"{CATALOG}.{SCHEMA}.dq_log"
BRONZE_TABLE = f"{CATALOG}.bronze.bronze_trips"


## Load Bronze Table

Read the raw trip data from the bronze layer table into a DataFrame for processing.

In [0]:
fdf = spark.table(BRONZE_TABLE)

## Inspect Data Schema

Display the schema of the bronze table to understand column names and data types before cleaning.

In [0]:
fdf.printSchema()

root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: string (nullable = true)
 |-- trip_end_timestamp: string (nullable = true)
 |-- trip_seconds: double (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_census_tract: double (nullable = true)
 |-- dropoff_census_tract: double (nullable = true)
 |-- pickup_community_area: double (nullable = true)
 |-- dropoff_community_area: double (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- company: string (nullable = true)
 |-- pickup_centroid_latitude: double (nullable = true)
 |-- pickup_centroid_longitude: double (nullable = true)
 |-- pickup_centroid_location: string (nullable = true)
 |-- dropoff_centroid_latitude: double (nullable = true)
 |-- dropoff_centroid

## Data Quality Assessment - Before Cleaning

Generate a data quality report on the raw bronze data, counting:
* Total rows
* Null fare values
* Null trip_miles values
* Negative fare amounts
* Zero or negative trip miles
* Missing timestamps
* Duplicate trip_id records

In [0]:
total_rows = fdf.count()
null_fare = fdf.filter(F.col("fare").isNull()).count()
null_miles = fdf.filter(F.col("trip_miles").isNull()).count()
neg_fare = fdf.filter(F.col("fare") < 0).count()
zero_miles = fdf.filter(F.col("trip_miles") <= 0).count()
no_ts = fdf.filter(F.col("trip_start_timestamp").isNull()).count()
dupes = fdf.groupBy("trip_id").count().filter("count >1").count()

print(f"""
      DQ report - bronze, before clean
      total rows: {total_rows}
      null fare: {null_fare}
      null miles: {null_miles}
      negative fare: {neg_fare}
      zero miles: {zero_miles}
      no timestamp: {no_ts}
      duplicates: {dupes}
      """)


      DQ report - bronze, before clean
      total rows: 2017139
      null fare: 2004
      null miles: 7
      negative fare: 0
      zero miles: 212701
      no timestamp: 0
      duplicates: 334
      


## Apply Cleaning Transformations

Clean and enrich the bronze data by:
* Casting columns to appropriate data types (DoubleType, TimestampType, StringType)
* Dropping rows with null values in critical columns (trip_id, fare, trip_miles, trip_start_timestamp)
* Filtering out invalid business records (fare ≤ 0, trip_miles ≤ 0, trip_seconds ≤ 0)
* Removing duplicate trip_id records
* Adding derived columns: pickup_hour, pickup_date, fare_per_mile, year_month
* Adding silver_processed_at audit timestamp

Report the number of clean rows and rejection rate.

In [0]:
# Applying cleaning transforms

dfclean = ( 
    fdf
    .withColumn("fare", F.col("fare").cast(DoubleType()))
    .withColumn("trip_miles", F.col("trip_miles").cast(DoubleType()))
    .withColumn("trip_seconds", F.col("trip_seconds").cast(DoubleType()))
    .withColumn("trip_start_timestamp", F.col("trip_start_timestamp").cast(TimestampType()))
    .withColumn("trip_end_timestamp", F.col("trip_end_timestamp").cast(TimestampType()))
    .withColumn("taxi_id", F.col("taxi_id").cast(StringType()))
    
    # Drop nulls for business critical columns
    .dropna(subset=["trip_id", "fare", "trip_miles", "trip_start_timestamp"])

    # Drop rows violating business rules
    .filter(F.col("fare")>0)
    .filter(F.col("trip_miles")>0)
    .filter(F.col("trip_seconds")>0)

    # Deduplicate
    .dropDuplicates(["trip_id"])
    .withColumn("pickup_hour",  F.hour("trip_start_timestamp"))
    .withColumn("pickup_date", F.to_date("trip_start_timestamp"))
    .withColumn("fare_per_mile",    F.round(F.col("fare")/F.col("trip_miles"),2))
    .withColumn("year_month", F.date_format(F.col("trip_start_timestamp"), "yyyy-MM"))
    
    # Add silver audit column
    .withColumn("silver_processed_at", F.current_timestamp())
)
clean_rows = dfclean.count()
rows_dropped = total_rows - clean_rows

print(f"""
DQ Report — After cleaning:
  Clean rows:    {clean_rows:>10,}
  Rows dropped:  {rows_dropped:>10,}  ({rows_dropped/total_rows*100:.1f}% rejection rate)
""")


DQ Report — After cleaning:
  Clean rows:     1,799,650
  Rows dropped:     217,489  (10.8% rejection rate)



## Log Data Quality Metrics

Create a data quality record with all metrics (null counts, duplicates, rejection rate) and append it to the DQ log table for tracking and monitoring over time.

In [0]:
dq_record = spark.createDataFrame([{
    "run_timestamp":      datetime.datetime.utcnow().isoformat(),
    "source_table":       "bronze_trips",
    "target_table":       "silver_trips",
    "total_rows_in":      total_rows,
    "null_fare_count":    null_fare,
    "null_miles_count":   null_miles,
    "negative_fare_count": neg_fare,
    "zero_miles_count":   zero_miles,
    "duplicate_count":    dupes,
    "rows_passed":        clean_rows,
    "rows_rejected":      rows_dropped,
    "rejection_rate_pct": round(rows_dropped / total_rows * 100, 2)
}])

(
    dq_record.write
    .format("delta")
    .mode("append")
    .saveAsTable(DQ_TABLE)
)


/home/spark-49d719c4-73cf-4b66-8e3e-17/.ipykernel/341/command-8874319575715804-2952006089:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_timestamp":      datetime.datetime.utcnow().isoformat(),


## Write Cleaned Data to Silver Table

Persist the cleaned data to the silver layer:
* If the table exists: perform a MERGE operation (upsert) based on trip_id
* If the table doesn't exist: create it with partitioning by year_month
* Optimize the table for query performance
* Display final row count

In [0]:
table_exists = spark.catalog.tableExists(SILVER_TABLE)

if table_exists:
    silver_table = DeltaTable.forName(spark, SILVER_TABLE)
    (
        silver_table.alias("target")
        .merge(
            dfclean.alias("source"),
            "target.trip_id = source.trip_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Silver table: MERGE INTO completed")
else:
    (
        dfclean.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("year_month")
        .saveAsTable(SILVER_TABLE)
    )
    print("Silver table: WRITE AS TABLE completed")

spark.sql(f"OPTIMIZE {SILVER_TABLE}")
print(f"Silver row count: {spark.table(SILVER_TABLE).count():,}")

Silver table: WRITE AS TABLE completed
Silver row count: 1,799,650
